# Dashboard (Phase 5)

This notebook provides a **read-only live dashboard** for simulation topics.

Phase 5 scope:
- Subscribe to raw simulation topics via MQTT
- Render live person markers on an anymap-ts map
- Show city-level indicators from observer snapshots
- Keep dashboard read-only (no simulation authority)

In [1]:
# Cell purpose: Import modules, load config, and define dashboard topic subscriptions.
from __future__ import annotations

import json
from collections import deque
from time import sleep

import simulated_city.mqtt as mqtt
from simulated_city.config import load_config
from simulated_city.maplibre_live import LiveMapLibreMap

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 5 requires a 'simulation' section in config.yaml")

base_topic = mqtt_cfg.base_topic
topic_root = f"{base_topic}/pandemic/#"
topic_trigger = f"{base_topic}/pandemic/trigger/person_state"
topic_exposure = f"{base_topic}/pandemic/observer/exposure_event"
topic_snapshot = f"{base_topic}/pandemic/observer/city_snapshot"

center_lnglat = (sim_cfg.city_center_lon, sim_cfg.city_center_lat)
print(f"Dashboard config loaded. center={center_lnglat}, subscribe={topic_root}")

Dashboard config loaded. center=(12.5683, 55.6761), subscribe=simulated-city/pandemic/#


In [2]:
# Cell purpose: Create anymap-ts map and initialize dashboard state containers.
dashboard_map = LiveMapLibreMap(center=center_lnglat, zoom=13, height="620px", width="100%")
dashboard_map.add_basemap("OpenStreetMap.Mapnik")

marker_ids: dict[str, str] = {}
marker_statuses: dict[str, str] = {}
messages_by_topic: dict[str, int] = {
    "trigger": 0,
    "exposure": 0,
    "snapshot": 0,
}
latest_city_snapshot: dict = {}
last_rendered_step = -1
incoming_messages = deque()

def marker_color(health_status: str) -> str:
    normalized_status = health_status.strip().lower()
    if normalized_status == "infected":
        return "#d32f2f"
    if normalized_status == "recovered":
        return "#2e7d32"
    return "#1565c0"

dashboard_map

In [3]:
# Cell purpose: Connect dashboard MQTT client, subscribe to raw topics, and queue incoming messages.
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="dashboard")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")

def on_message(client_obj, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode())
    except Exception as exc:
        print(f"Dashboard payload decode error: {exc}")
        return

    incoming_messages.append((msg.topic, payload))

client.on_message = on_message
client.subscribe(topic_root)
print(f"Subscribed to {topic_root}")
print("Dashboard is live. Run Trigger + Observer notebooks, then run the processing cell.")

Connected to MQTT broker at 127.0.0.1:1883
Subscribed to simulated-city/pandemic/#
Dashboard is live. Run Trigger + Observer notebooks, then run the processing cell.


In [ ]:
# Cell purpose: Process queued MQTT messages and update map/indicators on the notebook thread.
def process_dashboard_messages(max_items: int = 500) -> int:
    global last_rendered_step
    processed = 0

    while incoming_messages and processed < max_items:
        topic, payload = incoming_messages.popleft()
        processed += 1

        if topic == topic_trigger:
            messages_by_topic["trigger"] += 1
            person_id = str(payload.get("person_id", ""))
            if not person_id:
                continue
            try:
                lat = float(payload.get("lat"))
                lon = float(payload.get("lon"))
            except (TypeError, ValueError):
                continue
            status = str(payload.get("health_status", "susceptible")).strip().lower()
            popup_text = f"{person_id} | {status}"

            previous_status = marker_statuses.get(person_id)
            marker_id = marker_ids.get(person_id)

            if marker_id is None:
                marker_id = dashboard_map.add_marker(
                    lon,
                    lat,
                    name=person_id,
                    color=marker_color(status),
                    popup=popup_text,
                )
                marker_ids[person_id] = marker_id
            elif previous_status != status:
                try:
                    dashboard_map.remove_marker(marker_id)
                except Exception:
                    pass
                marker_id = dashboard_map.add_marker(
                    lon,
                    lat,
                    name=person_id,
                    color=marker_color(status),
                    popup=popup_text,
                )
                marker_ids[person_id] = marker_id
            else:
                dashboard_map.move_marker(
                    marker_id,
                    (lon, lat),
                    color=marker_color(status),
                    popup=popup_text,
                )

            marker_statuses[person_id] = status

            step = int(payload.get("step", 0))
            if step <= 2 and step != last_rendered_step:
                last_rendered_step = step
                print(f"Dashboard render step={step:04d} trigger_messages={messages_by_topic['trigger']}")

        elif topic == topic_exposure:
            messages_by_topic["exposure"] += 1

        elif topic == topic_snapshot:
            messages_by_topic["snapshot"] += 1
            latest_city_snapshot.clear()
            latest_city_snapshot.update(payload)
            step = int(payload.get("step", 0))
            if step <= 2:
                print(
                    f"Dashboard snapshot step={step:04d} infected={payload.get('infected_count')} "
                    f"susceptible={payload.get('susceptible_count')} exposures={payload.get('active_exposures')}"
                )

    return processed

print("Processing loop started. Interrupt the cell to stop.")
try:
    while True:
        processed_now = process_dashboard_messages(max_items=500)
        if processed_now == 0:
            sleep(0.05)
except KeyboardInterrupt:
    print("Processing loop stopped.")

Processing loop started. Interrupt the cell to stop.
Dashboard render step=0000 trigger_messages=1
Dashboard snapshot step=0000 infected=5 susceptible=195 exposures=0
Dashboard render step=0002 trigger_messages=201
Dashboard snapshot step=0002 infected=5 susceptible=195 exposures=1


In [ ]:
# Cell purpose: Print dashboard status counters and disconnect cleanly when finished.
print(
    f"Dashboard status: trigger_messages={messages_by_topic['trigger']}, "
    f"exposure_messages={messages_by_topic['exposure']}, "
    f"snapshot_messages={messages_by_topic['snapshot']}, "
    f"markers={len(marker_ids)}"
)
print(f"Latest snapshot: {latest_city_snapshot if latest_city_snapshot else 'none'}")

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")